In [6]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the dataset
path = "C:/Users/swagath/Desktop/VERACITY PYTHON AI COURSE/Assignment_2/healthcare_data.csv"
df = pd.read_csv(path)
print(df.head())
# Part A: Tricky Data Cleaning and Calculations

# Q1. Data Cleaning Challenge
def clean_cost_discount(df):
    """
    Convert Cost and Discount columns to numeric.
    Fill missing Cost with median Cost for each Diagnosis.
    Fill missing Discount with 0.
    """
    # Convert to numeric, coerce errors to NaN
    df['Cost'] = pd.to_numeric(df['Cost'], errors='coerce')
    df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce')
    
    # Fill missing discounts with 0 (no discount)
    df['Discount'] = df['Discount'].fillna(0)
    
    # Fill missing costs with median cost per Diagnosis
    df['Cost'] = df.groupby('Diagnosis')['Cost'].transform(lambda x: x.fillna(x.median()))
    return df

df = clean_cost_discount(df)

# Q2. Remove Hidden Duplicate Patients
def standardize_and_remove_duplicates(df):
    """
    Strip extra spaces from patient names and remove duplicates
    based on same Name, Age, and Diagnosis.
    Returns cleaned dataframe and count of duplicates removed.
    """
    # Remove leading/trailing spaces in 'Name'
    df['Name'] = df['Name'].str.strip()
    
    # Count before removing duplicates
    before = len(df)
    
    # Remove duplicates with same Name, Age, Diagnosis
    df = df.drop_duplicates(subset=['Name', 'Age', 'Diagnosis'])
    
    # Count how many records removed
    removed = before - len(df)
    return df, removed

df, num_duplicates_removed = standardize_and_remove_duplicates(df)

# Q3. Revenue Calculation
def calculate_final_bill(df):
    """
    Create a new column Final_Bill using formula:
    Final_Bill = Cost * (1 - Discount)
    Identify patient with maximum Final_Bill.
    """
    df['Final_Bill'] = df['Cost'] * (1 - df['Discount'])
    
    # Patient with max Final_Bill
    max_bill_row = df.loc[df['Final_Bill'].idxmax(), ['Name', 'Diagnosis', 'Doctor', 'Final_Bill']]
    return df, max_bill_row

df, max_bill_patient = calculate_final_bill(df)

# Part B: Medium Complexity Analytics

# Q4. Disease & Cost Analysis
def diagnosis_analysis(df):
    """
    For each Diagnosis, calculate:
    - Total patients
    - Average Final_Bill
    Find the disease with the highest average Final_Bill.
    """
    total_patients = df.groupby('Diagnosis').size().rename('Total_Patients')
    avg_final_bill = df.groupby('Diagnosis')['Final_Bill'].mean().rename('Avg_Final_Bill')
    
    diag_summary = pd.concat([total_patients, avg_final_bill], axis=1)
    most_expensive_disease = diag_summary['Avg_Final_Bill'].idxmax()
    
    return diag_summary, most_expensive_disease

diagnosis_summary, most_expensive_disease = diagnosis_analysis(df)

# Q5. Doctor Performance
def doctor_performance(df):
    """
    For each doctor, calculate:
    - Number of patients treated
    - Total revenue generated (sum of Final_Bill)
    Find the doctor with the highest revenue.
    """
    patients_treated = df.groupby('Doctor').size().rename('Patients_Treated')
    total_revenue = df.groupby('Doctor')['Final_Bill'].sum().rename('Total_Revenue')
    
    doctor_summary = pd.concat([patients_treated, total_revenue], axis=1)
    top_doctor = doctor_summary['Total_Revenue'].idxmax()
    
    return doctor_summary, top_doctor

doctor_summary, top_doctor = doctor_performance(df)

# Part C: Insights

# Q6. Age vs Disease Trend
def age_disease_trend(df):
    """
    Group patients into age groups:
    0-18, 19-35, 36-60, 61+
    For each group, find the most common diagnosis.
    """
    bins = [0, 18, 35, 60, np.inf]
    labels = ['0-18', '19-35', '36-60', '61+']
    
    df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True)

    # Most common diagnosis per age group
    most_common_disease = df.groupby('Age_Group', observed=True)['Diagnosis'].agg(lambda x: x.mode().iloc[0])
    
    return most_common_disease

age_group_diagnosis = age_disease_trend(df)

# Q7. Gender Bias in Treatment
def gender_bias_analysis(df):
    """
    Compute average Final_Bill by Gender.
    Determine which gender incurs higher costs on average.
    """
    avg_bill_gender = df.groupby('Gender')['Final_Bill'].mean()
    higher_cost_gender = avg_bill_gender.idxmax()
    return avg_bill_gender, higher_cost_gender

avg_bill_by_gender, gender_with_higher_cost = gender_bias_analysis(df)

# Q8. Seasonal Patient Flow
def monthly_patient_flow(df):
    """
    Extract month from Visit_Date and count patients per month.
    Find the busiest month.
    """
    df['Visit_Date'] = pd.to_datetime(df['Visit_Date'])
    
    df['Visit_Month'] = df['Visit_Date'].dt.month
    
    patient_counts = df['Visit_Month'].value_counts().sort_index()
    busiest_month = patient_counts.idxmax()
    
    return patient_counts, busiest_month

patients_per_month, busiest_month = monthly_patient_flow(df)

# -----------------------------
# Output all results for clarity

print(f"Duplicates Removed: {num_duplicates_removed}\n")

print("Patient with Maximum Final Bill:")
print(max_bill_patient)
print()

print("Diagnosis Summary (Total Patients and Average Final Bill):")
print(diagnosis_summary)
print(f"\nDisease with highest average final bill: {most_expensive_disease}\n")

print("Doctor Performance (Patients Treated and Total Revenue):")
print(doctor_summary)
print(f"\nTop Doctor by Revenue: {top_doctor}\n")

print("Most Common Diagnosis per Age Group:")
print(age_group_diagnosis)
print()

print("Average Final Bill by Gender:")
print(avg_bill_by_gender)
print(f"Gender with higher average treatment cost: {gender_with_higher_cost}\n")

print("Patient Counts per Month:")
print(patients_per_month)
print(f"Busiest Month (by patient count): {busiest_month}")


   Patient_ID     Name  Age  Gender      Diagnosis   Treatment       Doctor  \
0           1   Alice    52  Female  Heart Disease     Surgery  Dr. Johnson   
1           2   James    72  Female       Diabetes  Medication    Dr. Smith   
2           3  Olivia    83    Male      Arthritis     Surgery      Dr. Lee   
3           4      Bob   24  Female            Flu     Surgery  Dr. Johnson   
4           5   Alice     2    Male       Covid-19     Therapy   Dr. Wilson   

       Cost  Discount  Visit_Date  
0  19019.21       NaN  2024-04-16  
1  12013.30       NaN  2024-04-12  
2   2089.50       0.0  2024-03-28  
3  13052.68       0.1  2024-11-04  
4  14467.78       0.0  2024-10-20  
Duplicates Removed: 2

Patient with Maximum Final Bill:
Name                  Alice
Diagnosis     Heart Disease
Doctor            Dr. Smith
Final_Bill         19462.91
Name: 95, dtype: object

Diagnosis Summary (Total Patients and Average Final Bill):
               Total_Patients  Avg_Final_Bill
Diagnosis  